# 🔍 Probar Imágenes Propias con el Modelo MLP
## Predicciones individuales con visualizaciones para la presentación

Este notebook:
1. Carga el modelo MLP entrenado
2. Carga las imágenes propias preprocesadas
3. Muestra la predicción de **cada imagen individual** con gráfica de confianza
4. Genera un resumen visual completo

---
## 1. Importar Librerías y Cargar Modelo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import joblib
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 18

# Cargar modelo
mlp = joblib.load('../modelo/mlp_mnist.pkl')
print(f'✅ Modelo cargado')
print(f'   Capas ocultas: {mlp.hidden_layer_sizes}')
print(f'   Clases: {mlp.classes_}')

---
## 2. Cargar Imágenes Propias

In [ ]:
PROC_DIR = Path('../mis_imagenes_procesadas')
RAW_DIR = Path('../mis_imagenes/raw')

# Leer CSV de labels
csv_path = PROC_DIR / 'labels.csv'
if not csv_path.exists():
    print('❌ No se encontró labels.csv. Ejecuta primero el notebook 02.')
else:
    df = pd.read_csv(csv_path)
    print(f'✅ Cargadas {len(df)} imágenes propias')
    print(f'\n📊 Distribución:')
    print(df['label'].value_counts().sort_index().to_string())
    
    # Cargar imágenes como arrays
    images = []
    labels = []
    filenames = []
    originals = []
    
    for _, row in df.iterrows():
        proc_path = row['processed_path']
        orig_path = row['original_path']
        
        img = np.array(Image.open(proc_path).convert('L'), dtype=np.float32)
        images.append(img)
        labels.append(row['label'])
        filenames.append(row['filename'])
        originals.append(orig_path)
    
    images = np.array(images)
    labels = np.array(labels)
    
    # Normalizar a [0, 1] y aplanar a 784
    X_custom = images.reshape(len(images), -1) / 255.0
    
    print(f'\n   Shape de datos: {X_custom.shape}')
    print(f'   Rango: [{X_custom.min():.2f}, {X_custom.max():.2f}]')

---
## 3. Predicciones del Modelo

In [ ]:
# Predecir
y_pred = mlp.predict(X_custom)
y_proba = mlp.predict_proba(X_custom)

# Resultados rápidos
correct = (y_pred == labels)
acc = accuracy_score(labels, y_pred)

print(f'📈 Resultados Generales:')
print(f'   Correctas: {correct.sum()} de {len(labels)}')
print(f'   Incorrectas: {(~correct).sum()} de {len(labels)}')
print(f'   Accuracy: {acc*100:.1f}%')

---
## 4. 🎯 Predicción Individual de CADA Imagen

Para cada imagen se muestra:
- La imagen original y la procesada
- La predicción del modelo
- La confianza (probabilidad) para cada dígito

In [ ]:
for i in range(len(images)):
    fig = plt.figure(figsize=(16, 5))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 2.5])
    
    pred = y_pred[i]
    real = labels[i]
    proba = y_proba[i]
    is_correct = pred == real
    
    status = '✅ CORRECTO' if is_correct else '❌ INCORRECTO'
    status_color = '#2e7d32' if is_correct else '#c62828'
    
    # Imagen original
    ax0 = fig.add_subplot(gs[0])
    try:
        orig = Image.open(originals[i]).convert('L')
        ax0.imshow(orig, cmap='gray')
    except:
        ax0.imshow(images[i], cmap='gray')
    ax0.set_title('Original', fontsize=14)
    ax0.axis('off')
    
    # Imagen procesada (28x28)
    ax1 = fig.add_subplot(gs[1])
    ax1.imshow(images[i], cmap='gray')
    ax1.set_title('Procesada (28x28)', fontsize=14)
    ax1.axis('off')
    
    # Barplot de probabilidades
    ax2 = fig.add_subplot(gs[2])
    colors_bar = ['#c62828' if d == pred and not is_correct else
                  '#2e7d32' if d == pred and is_correct else
                  '#bbdefb' for d in range(10)]
    bars = ax2.barh(range(10), proba * 100, color=colors_bar, edgecolor='gray', linewidth=0.5)
    ax2.set_yticks(range(10))
    ax2.set_yticklabels([str(d) for d in range(10)], fontsize=14)
    ax2.set_xlabel('Confianza (%)', fontsize=14)
    ax2.set_xlim(0, 105)
    ax2.invert_yaxis()
    
    # Agregar porcentajes en las barras
    for d, bar in enumerate(bars):
        width = bar.get_width()
        if width > 3:
            ax2.text(width - 1, bar.get_y() + bar.get_height()/2,
                     f'{width:.1f}%', ha='right', va='center', fontsize=11,
                     fontweight='bold', color='white')
    
    ax2.set_title('Probabilidad por dígito', fontsize=14)
    
    # Título general
    fig.suptitle(
        f'{filenames[i]}  —  Predicción: {pred}  |  Real: {real}  —  {status}',
        fontsize=18, fontweight='bold', color=status_color, y=1.03
    )
    
    plt.tight_layout()
    
    # Guardar cada predicción individual
    fig.savefig(f'../resultados/prediccion_{filenames[i].split(".")[0]}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print()

---
## 5. 📊 Grid Resumen de TODAS las Predicciones

In [ ]:
n = len(images)
cols = min(8, n)
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 3.5 * rows))
fig.suptitle(f'Resumen: {correct.sum()}/{n} correctas ({acc*100:.1f}%)',
             fontsize=24, fontweight='bold', y=1.02)

if rows == 1 and cols == 1:
    axes = np.array([[axes]])
elif rows == 1:
    axes = axes.reshape(1, -1)
elif cols == 1:
    axes = axes.reshape(-1, 1)

for i in range(n):
    r, c = i // cols, i % cols
    ax = axes[r, c]
    ax.imshow(images[i], cmap='gray')
    
    is_ok = y_pred[i] == labels[i]
    symbol = '✅' if is_ok else '❌'
    color = 'green' if is_ok else 'red'
    
    ax.set_title(f'{symbol} Pred:{y_pred[i]} Real:{labels[i]}',
                 fontsize=11, color=color, fontweight='bold')
    ax.axis('off')

# Ocultar ejes vacíos
for i in range(n, rows * cols):
    r, c = i // cols, i % cols
    axes[r, c].axis('off')

plt.tight_layout()
plt.savefig('../resultados/resumen_todas_predicciones.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Guardado en resultados/resumen_todas_predicciones.png')

---
## 6. Matriz de Confusión — Imágenes Propias

In [ ]:
# Solo mostrar la matriz si hay más de 1 clase
unique_labels = np.unique(np.concatenate([labels, y_pred]))

fig, ax = plt.subplots(figsize=(10, 8))

cm = confusion_matrix(labels, y_pred, labels=range(10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=range(10), yticklabels=range(10),
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 14}, ax=ax)

ax.set_xlabel('Predicción', fontsize=18)
ax.set_ylabel('Valor Real', fontsize=18)
ax.set_title(f'Matriz de Confusión — Imágenes Propias\nAccuracy: {acc*100:.1f}%',
             fontsize=20, fontweight='bold')

plt.tight_layout()
plt.savefig('../resultados/confusion_propias.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Guardado en resultados/confusion_propias.png')

---
## 7. Reporte Detallado

In [ ]:
print('📋 Reporte de Clasificación — Imágenes Propias:\n')
print(classification_report(labels, y_pred, zero_division=0, digits=3))

---
## 8. Análisis: ¿Qué dígitos le cuestan más?

In [ ]:
# Accuracy por dígito
print('📊 Accuracy por dígito en imágenes propias:\n')
digit_results = []

for d in range(10):
    mask = labels == d
    if mask.sum() > 0:
        digit_acc = (y_pred[mask] == labels[mask]).mean()
        digit_results.append({
            'digito': d,
            'total': mask.sum(),
            'correctas': (y_pred[mask] == labels[mask]).sum(),
            'accuracy': digit_acc
        })
        status = '✅' if digit_acc == 1.0 else '⚠️' if digit_acc >= 0.5 else '❌'
        print(f'   {status} Dígito {d}: {digit_acc*100:.0f}% ({(y_pred[mask] == labels[mask]).sum()}/{mask.sum()})')

# Gráfica de accuracy por dígito
if len(digit_results) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    digits = [r['digito'] for r in digit_results]
    accs = [r['accuracy'] * 100 for r in digit_results]
    totals = [r['total'] for r in digit_results]
    
    colors = ['#2e7d32' if a == 100 else '#f57c00' if a >= 50 else '#c62828' for a in accs]
    bars = ax.bar(digits, accs, color=colors, edgecolor='black', linewidth=0.5)
    
    for bar, total, accuracy in zip(bars, totals, accs):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{accuracy:.0f}%\n(n={total})', ha='center', va='bottom',
                fontsize=12, fontweight='bold')
    
    ax.set_xlabel('Dígito', fontsize=16)
    ax.set_ylabel('Accuracy (%)', fontsize=16)
    ax.set_title('Accuracy por Dígito — Imágenes Propias', fontsize=20, fontweight='bold')
    ax.set_xticks(digits)
    ax.set_ylim(0, 115)
    ax.axhline(y=acc*100, color='blue', linestyle='--', linewidth=2, alpha=0.5,
               label=f'Accuracy promedio: {acc*100:.1f}%')
    ax.legend(fontsize=14)
    
    plt.tight_layout()
    plt.savefig('../resultados/accuracy_por_digito.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('💾 Guardado en resultados/accuracy_por_digito.png')

---
## 9. Tabla de Confianza para Cada Predicción

In [ ]:
# Crear tabla con resultados detallados
results_df = pd.DataFrame({
    'Archivo': filenames,
    'Real': labels,
    'Predicción': y_pred,
    'Correcto': ['✅' if c else '❌' for c in correct],
    'Confianza (%)': [f'{y_proba[i][y_pred[i]]*100:.1f}%' for i in range(len(y_pred))],
    'Confianza Real (%)': [f'{y_proba[i][labels[i]]*100:.1f}%' for i in range(len(labels))]
})

print('📋 Tabla de Resultados Detallados:\n')
print(results_df.to_string(index=False))

---
## 10. Conclusiones

In [ ]:
print('=' * 60)
print('  RESUMEN FINAL')
print('=' * 60)
print(f'\n🧠 Modelo: MLP (128, 64) con scikit-learn')
print(f'📊 Dataset: MNIST (70,000 imágenes)')
print(f'\n📈 Resultados en MNIST test: ~97%+ accuracy')
print(f'📈 Resultados en imágenes propias: {acc*100:.1f}% accuracy')
print(f'\n   Total imágenes propias: {len(labels)}')
print(f'   Correctas: {correct.sum()}')
print(f'   Incorrectas: {(~correct).sum()}')

if (~correct).sum() > 0:
    print(f'\n⚠️  Imágenes que falló:')
    for i in range(len(labels)):
        if not correct[i]:
            print(f'   - {filenames[i]}: predijo {y_pred[i]} (era {labels[i]}, confianza: {y_proba[i][y_pred[i]]*100:.1f}%)')

print(f'\n💾 Todas las gráficas guardadas en: resultados/')
print(f'\n✅ ¡Listo para la presentación!')

---
## 11. Explicación de Resultados

### Comparación: MNIST test vs. imágenes propias
El modelo MLP fue entrenado **exclusivamente** con las 60,000 imágenes del conjunto de entrenamiento de MNIST y alcanzó un accuracy superior al 97% en las 10,000 imágenes de prueba estándar.

Al probarlo con nuestras imágenes propias (dibujadas a mano en papel y fotografiadas), es esperado que el accuracy sea **menor** que en el test de MNIST. Esto se debe a varias razones:

1. **Diferencia de dominio:** Las imágenes de MNIST fueron recopiladas de formularios escritos a mano con un proceso estandarizado, mientras que nuestras imágenes provienen de fotos de papel con condiciones de iluminación y ángulo variables.

2. **Preprocesamiento:** Aunque se aplicó un pipeline para convertir las fotos al formato MNIST (28×28, escala de grises, fondo negro, dígito blanco centrado), el resultado nunca será idéntico a la distribución original del dataset.

3. **Estilo de escritura:** Cada persona tiene una caligrafía distinta. El modelo aprendió patrones de muchos escritores diferentes en MNIST, pero puede tener dificultades con estilos muy particulares.

### Dígitos que más le cuestan al modelo
A partir de la gráfica de accuracy por dígito y la matriz de confusión, se puede observar cuáles dígitos fueron más difíciles de clasificar correctamente. Generalmente son dígitos que comparten rasgos visuales:
- **1 y 7** — ambos son trazos verticales con pequeñas diferencias
- **3 y 8** — comparten la forma curva superior
- **4 y 9** — la parte superior cerrada puede confundirse
- **5 y 6** — el trazo curvo inferior es similar

### Conclusión
El modelo MLP demuestra que una red neuronal relativamente simple puede aprender a clasificar dígitos escritos a mano con alta precisión. Sin embargo, al exponerlo a datos del mundo real (nuestras propias imágenes), queda claro que la **calidad del preprocesamiento** y la **similitud con los datos de entrenamiento** son factores críticos para el desempeño del modelo. Esto ilustra un concepto fundamental en machine learning: un modelo es tan bueno como los datos con los que fue entrenado y la representatividad de esos datos respecto a los datos reales.